# EXERKINEMAP sequence gym

Constrained variant design for exerkines, ligands and receptors. The notebook samples amino-acid substitutions in a protein with ESM-2 masked Gibbs sampling. Chosen regions stay fixed (for example the signal peptide, receptor-binding residues, cysteines or glycosylation sites). It then scores the variants by how natural they look to ESM-2.

**Workflow**
1. Fetch sequences live from UniProt (exerkines and ligands on one side, candidate receptors on the other).
2. Mark the fixed and mutable regions for each protein.
3. Generate variants with a masked Gibbs sampler over the 20 standard amino acids.
4. Report identity to wild type, the mutation list, and an ESM-2 pseudo-log-likelihood.

**What the results do not show.** The pseudo-log-likelihood is a plausibility proxy. It does not establish folding, binding or function. It is also circular when the same model generated the variants, so check important candidates with an independent structure predictor (ESMFold) and with experimental or database evidence.

**Before you rely on a result**
* Verify every UniProt accession. They were written from memory.
* ESM-2 accepts at most 1022 residues. Trim longer proteins to a domain first.

**Running it**
* Google Colab: Runtime > Change runtime type > GPU (optional). Install the dependencies first:

```bash
pip -q install torch transformers requests pandas
```

* Locally: select a kernel that has these packages installed. The default model, `esm2_t12_35M_UR50D`, runs on CPU. Switch to `esm2_t33_650M_UR50D` on a GPU.

In [1]:
import os
# Fall back to CPU for unsupported MPS operations instead of crashing
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

In [2]:
import re, random, requests, torch, pandas as pd
from transformers import AutoTokenizer, EsmForMaskedLM

MODEL = "facebook/esm2_t12_35M_UR50D"   # swap for esm2_t33_650M_UR50D on a GPU
dev = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(MODEL)
model = EsmForMaskedLM.from_pretrained(MODEL).eval().to(dev)
MAXLEN = 1022   # ESM-2 limit (1024 incl. <cls>/<eos>)

AA = "ACDEFGHIKLMNPQRSTVWY"
aa_ids = torch.tensor([tok.convert_tokens_to_ids(a) for a in AA], device=dev)  # only sample real amino acids
print(dev, "ready")

tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/778 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/136M [00:00<?, ?B/s]

cpu ready


## 1. Sequences
Human accessions (UniProt). Exerkines/ligands on the left, candidate receptors on the right. Edit freely.
Rat orthologs: use `fetch_by_gene("Il6", taxon=10116)`.

In [3]:
LIGANDS = {"IL6":"P05231","FGF21":"Q9NSA1","FNDC5_irisin_precursor":"Q8NAU1","GDF15":"Q99988",
           "BDNF":"P23560","IL15":"P40933","APLN":"Q9ULZ1","MSTN":"O14793"}
RECEPTORS = {"IL6R":"P08887","IL6ST":"P40189","KLB":"Q86Z14","FGFR1":"P11362","GFRAL":"Q6UXV0",
             "RET":"P07949","NTRK2":"Q16620","IL15RA":"Q13261","APLNR":"P35414","ACVR2B":"Q13705","ITGAV":"P06756"}

def _fasta_to_seq(txt):
    return "".join(l.strip() for l in txt.splitlines() if not l.startswith(">"))

def fetch(acc):
    r = requests.get(f"https://rest.uniprot.org/uniprotkb/{acc}.fasta", timeout=30)
    r.raise_for_status()
    return _fasta_to_seq(r.text)

def fetch_by_gene(gene, taxon=9606):
    q = f"gene_exact:{gene} AND organism_id:{taxon} AND reviewed:true"
    r = requests.get("https://rest.uniprot.org/uniprotkb/search",
                     params={"query": q, "format": "fasta", "size": 1}, timeout=30)
    r.raise_for_status()
    return _fasta_to_seq(r.text)

SEQS = {}
for name, acc in {**LIGANDS, **RECEPTORS}.items():
    try:
        SEQS[name] = fetch(acc)
    except Exception as e:
        print("failed", name, acc, e)
pd.DataFrame({"name": list(SEQS), "length": [len(s) for s in SEQS.values()],
              "fits_ESM2": [len(s) <= MAXLEN for s in SEQS.values()]})

,name,length,fits_ESM2
0,IL6,212,True
1,FGF21,209,True
2,FNDC5_irisin_precursor,260,True
3,GDF15,308,True
4,BDNF,247,True
5,IL15,162,True
6,APLN,77,True
7,MSTN,375,True
8,IL6R,468,True
9,IL6ST,918,True


## 2. Choose what to keep fixed
UniProt sequences are full precursors (signal peptide, propeptide, mature chain). Fix the regions you must not change:
signal peptide, receptor-binding residues, cysteines, glycosylation sites. Ranges are 1-based and inclusive.

In [4]:
def mutable_positions(seq, fixed_ranges=(), fix_cysteines=True, fix_first=0):
    fixed = set(range(fix_first))                       # e.g. fix_first=25 keeps a signal peptide
    for a, b in fixed_ranges:
        fixed |= set(range(a - 1, b))
    if fix_cysteines:
        fixed |= {i for i, c in enumerate(seq) if c == "C"}
    return torch.tensor([i for i in range(len(seq)) if i not in fixed], dtype=torch.long)

# example: IL6, keep the first 28 residues (signal peptide) and all cysteines
seq = SEQS["IL6"]
mutable = mutable_positions(seq, fix_first=28)
print(len(seq), "residues,", len(mutable), "mutable")

212 residues, 180 mutable


## 3. Masked Gibbs sampler
Differences from the quick sketch: logits are restricted to the 20 amino acids (so no `<cls>`, `<pad>` or `X` can be sampled),
and positions are drawn without replacement per sweep.

In [5]:
@torch.no_grad()
def gibbs(seq, mutable, steps=50, k=5, temp=0.8, seed=None):
    if len(seq) > MAXLEN:
        raise ValueError(f"sequence length {len(seq)} > {MAXLEN}; trim to a domain first")
    g = torch.Generator(device="cpu")
    if seed is not None: g.manual_seed(seed)
    ids = tok(seq, return_tensors="pt")["input_ids"].to(dev)    # residue i -> token i+1
    mutable = mutable.to("cpu")
    for _ in range(steps):
        pos = mutable[torch.randperm(len(mutable), generator=g)[:k]]
        masked = ids.clone()
        masked[0, pos.to(dev) + 1] = tok.mask_token_id
        logits = model(masked).logits[0, pos.to(dev) + 1][:, aa_ids] / temp   # (k, 20)
        pick = torch.multinomial(logits.softmax(-1).cpu(), 1, generator=g).squeeze(-1).to(dev)
        ids[0, pos.to(dev) + 1] = aa_ids[pick]
    return tok.decode(ids[0], skip_special_tokens=True).replace(" ", "")

def identity(a, b):
    return sum(x == y for x, y in zip(a, b)) / len(a)

def diff(a, b):
    return [f"{x}{i+1}{y}" for i, (x, y) in enumerate(zip(a, b)) if x != y]

In [6]:
variants = [gibbs(seq, mutable, steps=60, k=5, temp=0.8, seed=s) for s in range(8)]
rows = [{"variant": i, "identity_to_WT": round(identity(seq, v), 3), "n_mut": len(diff(seq, v)),
         "mutations": " ".join(diff(seq, v)[:12])} for i, v in enumerate(variants)]
pd.DataFrame(rows)

,variant,identity_to_WT,n_mut,mutations
0,0,0.358,136,P31A P32S G33Q E34D S36G D38E V39C A40P A41E P...
1,1,0.340,140,V30A P31F P32A E34P D35A S36P K37P D38I V39D A...
2,2,0.368,134,P29T P31A G33A E34V D35S K37E A40F A41K R44A Q...
3,3,0.349,138,P29A V30A P31L P32A G33Q E34S D35P S36G D38S V...
4,4,0.340,140,P29S V30G P31L P32E G33S E34S D35G K37D A40F P...
5,5,0.401,127,P29A V30A P31S P32S G33S E34A S36K D38A V39L A...
6,6,0.387,130,P29A V30G P31L G33E E34A D35G S36A K37R D38A V...
7,7,0.363,135,P29A V30A P31S P32E G33E E34V S36P K37D D38G A...


## 4. Quick plausibility score (pseudo-log-likelihood)
Masks each position in turn (L forward passes, so keep sequences short). Higher (less negative) = more plausible to ESM-2.
This is **not** evidence of function, and it is circular if the same model generated the variants; use ESMFold or other independent predictors too.

In [7]:
@torch.no_grad()
def pll(s):
    ids = tok(s, return_tensors="pt")["input_ids"].to(dev)
    tot = 0.0
    for i in range(1, ids.shape[1] - 1):
        m = ids.clone(); m[0, i] = tok.mask_token_id
        lp = model(m).logits[0, i].log_softmax(-1)
        tot += lp[ids[0, i]].item()
    return tot / (ids.shape[1] - 2)

print("WT   ", round(pll(seq), 3))
for i, v in enumerate(variants[:4]):
    print("var", i, round(pll(v), 3))

WT    -2.711
var 0 -2.635
var 1 -2.638
var 2 -2.755
var 3 -2.688


## 5. Next steps (optional)
* **Structure check:** fold WT and variants (ESMFold via the Hugging Face `EsmForProteinFolding`, or a hosted API) and compare backbone RMSD.
* **Novelty:** compare to the nearest natural sequence (BLAST/UniProt) and to your training set.
* **Receptor binding:** score ligand-receptor pairs with your protein-based compatibility model before and after mutation.
* **Rat vs human:** repeat with `fetch_by_gene(..., taxon=10116)` to see how ortholog differences change the outcome.